In [2]:
from pathlib import Path
curr_dir_parent = Path.cwd().parent

FILE_DIR = curr_dir_parent / "data/raw/intensities.csv"
DEM_FILE_PATH = curr_dir_parent / "gmted2010/mi30_grd/hdr.adf"

RSN,Event_Number,USGS_ComCatID,Atkinson_eventcode,CHIMP_eventcode,Source_of_Data,Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,Depth,Preferred_Magnitude,Source_of_Preferred_Magnitude,ComCat_Magnitude_Type,ComCat_Magnitude_Network,CHIMP_Magnitude_Type,NGAW3_Magnitude,NGAW3_Magnitude_Type,NGAW3_EventID,Regime,Sub_Regime,Record_Latitude,Record_Longitude,Intensity,Standard_Deviation,Nresp,Location,Rrup_Combined,Repi,Rhypo,Rjb_Combined,RrupFF,RJBFF,RxFF,RyFF,Ry0FF,Source_of_Distances,Thompson_VS30,Eon,Era,Period,Epoch,Division,Simplified_Unit,Unit,Lithology_1,Lithology_2,Reference,Digital_URL,NGMDB,NGAW3_MotionID,Closest_Station_Lat,Closest_Station_Long,Closest_Distance_km,PGA_g,PGV_cms,Duplicate,Max_Duplicate_Intensity_Difference,Average_Duplicated_Intensity,Unnamed: 63


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_ = pd.read_csv(FILE_DIR)

/tmp/ipykernel_10034/177482109.py:6: DtypeWarning: Columns (4: CHIMP_eventcode, 19: CHIMP_Magnitude_Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ = pd.read_csv(FILE_DIR)


In [14]:
column_map = {
"USGS_ComCatID": "id",
"Latitude":"lat",
"Longitude":"lng",
"Depth":"depth_sea",
"Preferred_Magnitude":"mag",
"Rhypo":"rhypo",
"Record_Latitude":"record_lat",
"Record_Longitude":"record_lng"
}
df = df_.filter(items=column_map.keys())
df = df.rename(columns=column_map)
df.dropna(inplace=True)
df

,id,lat,lng,depth_sea,mag,rhypo,record_lat,record_lng
18,nc71729135,38.79267,-122.7432,0.477,4.16,2.129752,38.80343,-122.7236
22,nc72737985,38.82217,-122.8413,1.480,5.01,2.670254,38.82164,-122.8157
31,ci3031111,34.20000,-116.4370,-0.097,7.30,10.555350,34.29491,-116.4350
42,nn00385392,39.52310,-119.9393,2.800,5.10,3.029873,39.51470,-119.9313
44,nc72737985,38.82217,-122.8413,1.480,5.01,2.307831,38.82170,-122.8618
...,...,...,...,...,...,...,...,...
695938,nc40226685,38.24150,-122.1773,9.295,3.11,9.295266,38.24213,-122.1772
695939,nc51209305,37.83650,-122.2153,12.466,3.05,12.466130,37.83682,-122.2158
695940,ci37764391,33.16967,-115.6162,8.650,3.37,8.650199,33.17013,-115.6165
695941,nc72542915,37.78100,-121.9668,9.240,3.21,9.240043,37.78082,-121.9666


In [15]:
import numpy as np
import pandas as pd
import rasterio


def compute_surface_depth(
	df: pd.DataFrame,
	dem_filepath: str,
	lon_col: str = "Longitude",
	lat_col: str = "Latitude",
	depth_col: str = "Depth",
	elevation_col: str = "elevation",
	depth_surface_col: str = "depth_surface",
	save_elevation: bool = True,
) -> pd.DataFrame:
	"""Samples elevation from a local DEM raster and calculates depth below ground surface.

	Parameters
	----------
	df : pd.DataFrame
		Input earthquake DataFrame.
	dem_filepath : str
		Path to the DEM file (GeoTIFF or ArcGrid header folder).
	lon_col : str, optional
		Column name for longitude coordinates, by default "Longitude".
	lat_col : str, optional
		Column name for latitude coordinates, by default "Latitude".
	depth_col : str, optional
		Column name for USGS depth in km, by default "Depth".

	Returns
	-------
	pd.DataFrame
		The modified DataFrame containing 'elevation' and
		'depth_surface'.
	"""
	# 1. Zip coordinate pairs (Longitude, Latitude)
	coords = list(zip(df[lon_col], df[lat_col]))

	# 2. Batch-sample DEM raster using memory-mapped C-speed iterator
	with rasterio.open(dem_filepath) as src:
		elevations_m = np.fromiter(
			(val[0] for val in src.sample(coords)), dtype=np.float32
		)

	# 3. Vectorized unit conversion (m to km) and depth computation
	df[elevation_col] = elevations_m / 1000.0
	df[depth_surface_col] = (df[elevation_col] + df[depth_col]).clip(
		lower=0.0
	).round(decimals=5)
	if not save_elevation:
		df.drop(columns=[elevation_col], inplace=True)
		
	return df

In [16]:
df = compute_surface_depth(
	df, DEM_FILE_PATH, lon_col="lng", 
	lat_col="lat", depth_col="depth_sea", 
	elevation_col="elevation", depth_surface_col="depth_surface_ele",save_elevation=False)
print(df.head())
display(df)

            id       lat       lng  depth_sea   mag      rhypo  record_lat  \
18  nc71729135  38.79267 -122.7432      0.477  4.16   2.129752    38.80343   
22  nc72737985  38.82217 -122.8413      1.480  5.01   2.670254    38.82164   
31   ci3031111  34.20000 -116.4370     -0.097  7.30  10.555350    34.29491   
42  nn00385392  39.52310 -119.9393      2.800  5.10   3.029873    39.51470   
44  nc72737985  38.82217 -122.8413      1.480  5.01   2.307831    38.82170   

    record_lng  depth_surface_ele  
18   -122.7236              1.522  
22   -122.8157              2.086  
31   -116.4350              0.957  
42   -119.9313              4.261  
44   -122.8618              2.086  


,id,lat,lng,depth_sea,mag,rhypo,record_lat,record_lng,depth_surface_ele
18,nc71729135,38.79267,-122.7432,0.477,4.16,2.129752,38.80343,-122.7236,1.522
22,nc72737985,38.82217,-122.8413,1.480,5.01,2.670254,38.82164,-122.8157,2.086
31,ci3031111,34.20000,-116.4370,-0.097,7.30,10.555350,34.29491,-116.4350,0.957
42,nn00385392,39.52310,-119.9393,2.800,5.10,3.029873,39.51470,-119.9313,4.261
44,nc72737985,38.82217,-122.8413,1.480,5.01,2.307831,38.82170,-122.8618,2.086
...,...,...,...,...,...,...,...,...,...
695938,nc40226685,38.24150,-122.1773,9.295,3.11,9.295266,38.24213,-122.1772,9.357
695939,nc51209305,37.83650,-122.2153,12.466,3.05,12.466130,37.83682,-122.2158,12.631
695940,ci37764391,33.16967,-115.6162,8.650,3.37,8.650199,33.17013,-115.6165,8.580
695941,nc72542915,37.78100,-121.9668,9.240,3.21,9.240043,37.78082,-121.9666,9.378


In [20]:
from math import radians, sin, cos, sqrt, atan2


EARTH_RADIUS_KM = 6371.0
def haversine_distance(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return EARTH_RADIUS_KM * c
print(haversine_distance(38.79267 , -112.7432 ,38.80343,-122.7236))  

def find_hav_dist(
    df: pd.DataFrame,
    lat_col: str = "lat",
    lon_col: str = "lng",
    record_lat_col: str = "record_lat",
    record_lon_col: str = "record_lng",
) -> pd.Series:
    """
    Compute Haversine distance between two coordinate pairs for each row.

    Returns:
        pd.Series containing distance in kilometers.
    """

    lat1 = np.radians(df[lat_col])
    lon1 = np.radians(df[lon_col])

    lat2 = np.radians(df[record_lat_col])
    lon2 = np.radians(df[record_lon_col])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a),
    )

    return EARTH_RADIUS_KM * c
    
    

864.4802370793855


In [ ]:
import numpy as np
# using coordinate to find depth from surface to hypocenter using haversine distance

def get_group_median(group):
  hav_distances = find_hav_dist(group)
  diff_sq = (group["rhypo"] ** 2) - (hav_distances ** 2)
  diff_sq = np.maximum(0, diff_sq)
  depth_sur_hav = np.sqrt(diff_sq)
  return depth_sur_hav.mean()

group_medians = df.groupby("id", group_keys=False).apply(get_group_median)

df["depth_surface_hv"] = df["id"].map(group_medians)

display(df)

,id,lat,lng,depth_sea,mag,rhypo,record_lat,record_lng,depth_surface_ele,depth_surface_hv
18,nc71729135,38.79267,-122.7432,0.477,4.16,2.129752,38.80343,-122.7236,1.522,0.430555
22,nc72737985,38.82217,-122.8413,1.480,5.01,2.670254,38.82164,-122.8157,2.086,1.517437
31,ci3031111,34.20000,-116.4370,-0.097,7.30,10.555350,34.29491,-116.4350,0.957,0.312920
42,nn00385392,39.52310,-119.9393,2.800,5.10,3.029873,39.51470,-119.9313,4.261,2.800631
44,nc72737985,38.82217,-122.8413,1.480,5.01,2.307831,38.82170,-122.8618,2.086,1.517437
...,...,...,...,...,...,...,...,...,...,...
695938,nc40226685,38.24150,-122.1773,9.295,3.11,9.295266,38.24213,-122.1772,9.357,9.295994
695939,nc51209305,37.83650,-122.2153,12.466,3.05,12.466130,37.83682,-122.2158,12.631,12.464898
695940,ci37764391,33.16967,-115.6162,8.650,3.37,8.650199,33.17013,-115.6165,8.580,8.650341
695941,nc72542915,37.78100,-121.9668,9.240,3.21,9.240043,37.78082,-121.9666,9.378,9.237691


### Findings & Decisions (`depth_eval.ipynb`)

* **Feature Selection (`depth_surface_ele` vs. `depth_surface_hv`):**
* `depth_surface_hv` (calculated using Haversine horizontal distance and `rhypo`) lacks necessary accuracy.
* `depth_surface_ele` (calculated using DEM elevation data) is significantly more accurate and reliable.
* **Decision:** We will use `depth_surface_ele` as the primary hypocentral surface depth metric for model training and evaluation.


* **Data Pipeline Trade-off:**
* Incorporating DEM elevation data adds approximately 500 MB to the project dataset.
* This storage trade-off is acceptable because automated retrieval is already configured in `data_download.ipynb`, and the substantial accuracy gains outweigh the additional data footprint.